# 01 · Data Extraction & Validation
**Core question:** *What share of Blinkit's sales is impulse vs planned, and what convenience premium do users pay?*

This notebook loads the raw dataset, inspects its structure, checks data quality, and frames initial hypotheses grounded in the data.

In [1]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('../data/raw/blinkit_dataset (2).csv')
print('Loaded:', df.shape)

Loaded: (13000, 25)


### 2 · Shape, Columns & Sample Rows

In [2]:
print('Rows:', df.shape[0], '| Columns:', df.shape[1])
print('\nColumn list:', df.columns.tolist())
df.head()

Rows: 13000 | Columns: 25

Column list: ['product_id', 'product_name', 'category', 'brand', 'price', 'discount_pct', 'final_price', 'rating', 'num_reviews', 'delivery_time_min', 'city', 'seller', 'stock', 'sold_quantity', 'profit_margin_pct', 'is_organic', 'packaging_type', 'weight_g', 'shelf_life_days', 'reorder_level', 'demand_index', 'date_added', 'expiry_date', 'offer_type', 'delivery_status']


,product_id,product_name,category,brand,price,discount_pct,final_price,rating,num_reviews,delivery_time_min,...,is_organic,packaging_type,weight_g,shelf_life_days,reorder_level,demand_index,date_added,expiry_date,offer_type,delivery_status
0,1,Tata Organic Grocery 300,Grocery,Tata,199.78,25,149.84,4.5,146,37,...,True,Can,750,212,15,73,2023-11-27,2024-06-26,NaN,On-Time
1,2,Mother Dairy Lite Dairy 275,Dairy,Mother Dairy,44.32,30,31.02,4.0,264,36,...,False,Jar,1000,17,24,25,2024-08-07,2024-08-24,NaN,Delayed
2,3,P&G Classic Personal 439,Personal Care,P&G,501.13,0,501.13,3.7,69,17,...,True,Jar,1000,1463,25,100,2024-03-03,2028-03-05,FreeDelivery,On-Time
3,4,Dettol Fresh Household 771,Household,Dettol,627.17,0,627.17,3.9,103,23,...,True,Bottle,200,1143,18,15,2024-08-07,2027-09-24,NaN,On-Time
4,5,Minute Maid Daily Beverages 264,Beverages,Minute Maid,101.69,15,86.44,4.3,422,10,...,True,Can,300,363,30,6,2024-07-04,2025-07-02,NaN,On-Time


### 3 · Data-Type Analysis

In [3]:
print(df.dtypes)
num = df.select_dtypes('number').columns.tolist()
cat = df.select_dtypes('object').columns.tolist()
print(f'\nNumerical ({len(num)}):', num)
print(f'Categorical ({len(cat)}):', cat)

product_id             int64
product_name             str
category                 str
brand                    str
price                float64
discount_pct           int64
final_price          float64
rating               float64
num_reviews            int64
delivery_time_min      int64
city                     str
seller                   str
stock                  int64
sold_quantity          int64
profit_margin_pct    float64
is_organic              bool
packaging_type           str
weight_g               int64
shelf_life_days        int64
reorder_level          int64
demand_index           int64
date_added               str
expiry_date              str
offer_type               str
delivery_status          str
dtype: object

Numerical (14): ['product_id', 'price', 'discount_pct', 'final_price', 'rating', 'num_reviews', 'delivery_time_min', 'stock', 'sold_quantity', 'profit_margin_pct', 'weight_g', 'shelf_life_days', 'reorder_level', 'demand_index']
Categorical (10): ['product_name

### 4 · Missing Values (highlight `offer_type`)
Pandas reads the string `None` in the CSV as actual NaN for `offer_type`.

In [4]:
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
result = pd.DataFrame({'count': miss, 'pct': miss_pct})
print(result[result['count'] > 0].to_string() if result['count'].sum() > 0 else 'No NaN values found.')
print(f"\noffer_type NaN count: {df['offer_type'].isna().sum()}  ({df['offer_type'].isna().mean()*100:.1f}%)")

            count    pct
offer_type   6544  50.34

offer_type NaN count: 6544  (50.3%)


### 5 · Data-Quality Checks

In [5]:
print('Negative prices       :', (df['final_price'] < 0).sum())
print('Zero/neg quantities   :', (df['sold_quantity'] <= 0).sum())
print('Invalid weights       :', (df['weight_g'] <= 0).sum())
cost_est = df['final_price'] * (1 - df['profit_margin_pct']/100)
print('final < est. cost     :', (df['final_price'] < cost_est).sum())

Negative prices       : 0
Zero/neg quantities   : 47
Invalid weights       : 0
final < est. cost     : 0


### 6 · Basic Distributions

In [6]:
print('-- Category distribution --')
print(df['category'].value_counts())
print('\n-- City distribution --')
print(df['city'].value_counts())
print('\n-- Price stats --')
print(df['final_price'].describe())

-- Category distribution --
category
Bakery                 1713
Grocery                1700
Dairy                  1662
Personal Care          1620
Snacks                 1596
Fruits & Vegetables    1588
Beverages              1561
Household              1560
Name: count, dtype: int64

-- City distribution --
city
Mumbai       1349
Bengaluru    1345
Lucknow      1320
Ahmedabad    1315
Pune         1309
Jaipur       1302
Hyderabad    1285
Delhi        1273
Chennai      1257
Kolkata      1245
Name: count, dtype: int64

-- Price stats --
count    13000.000000
mean       240.735212
std        182.405552
min          8.140000
25%        108.790000
50%        197.185000
75%        316.882500
max        998.920000
Name: final_price, dtype: float64


### 7 · Initial Business Hypotheses (data-driven)

In [7]:
cat_stats = df.groupby('category').agg(
    med_qty=('sold_quantity','median'),
    med_weight=('weight_g','median'),
    med_shelf=('shelf_life_days','median'),
    offer_share=('offer_type', lambda x: x.notna().mean()*100)
).sort_values('med_qty', ascending=False)
print('Category stats (sorted by median qty sold):')
print(cat_stats.round(1))

Category stats (sorted by median qty sold):
                     med_qty  med_weight  med_shelf  offer_share
category                                                        
Dairy                  124.0       750.0       19.0         49.0
Fruits & Vegetables    123.0       500.0        9.0         50.3
Personal Care          123.0       400.0     1104.0         52.7
Snacks                 123.0       300.0      216.5         49.4
Beverages              120.0       500.0      322.0         49.2
Household              117.5       300.0     1103.5         48.5
Grocery                114.0       400.0      460.5         50.4
Bakery                 112.0       300.0        8.0         47.9


In [8]:
has_offer = df['offer_type'].notna()
with_offer = df.loc[has_offer, 'sold_quantity'].mean()
without_offer = df.loc[~has_offer, 'sold_quantity'].mean()
print(f'Avg qty sold WITH offer  : {with_offer:.1f}')
print(f'Avg qty sold WITHOUT offer: {without_offer:.1f}')

Avg qty sold WITH offer  : 163.4
Avg qty sold WITHOUT offer: 161.5


In [9]:
small = df['weight_g'] < 300
print(f'Avg qty sold small packs (<300g) : {df.loc[small, "sold_quantity"].mean():.1f}')
print(f'Avg qty sold other packs (>=300g): {df.loc[~small, "sold_quantity"].mean():.1f}')

Avg qty sold small packs (<300g) : 164.1
Avg qty sold other packs (>=300g): 161.5
